In [ ]:
import numpy as np 
import skimage
import matplotlib.pyplot as plt
from pathlib import Path 
import tifffile as tf 
import pickle 
import cv2
from matplotlib_scalebar.scalebar import ScaleBar
import scipy

In [ ]:
PAS = True

In [ ]:
key = '' # sample identifier

In [ ]:
registration_base_path = r"" # path to registered images 

In [ ]:
pixel_barcode_base_path = r"" # path to folder with cell pixel barcode results

In [ ]:
HE_registered = tf.imread(registration_base_path + '/H&E_registered.tif')
print('HE read')
if(PAS):
    PAS_registered = tf.imread(registration_base_path + '/PAS_registered.tif')
    print('PAS read')
Trichrome_registered = tf.imread(registration_base_path + '/Trichrome_registered.tif')
print("Trichrome read")

In [ ]:
if(PAS):
    stains = ['H&E','PAS','Trichrome']
else:
    stains = ['H&E', 'Trichrome']

In [ ]:
if(PAS):
    stain_images = [HE_registered, PAS_registered, Trichrome_registered]
else:
    stain_images = [HE_registered, Trichrome_registered]

In [ ]:
with open(registration_base_path + '/roi_coord_dict.pickle', 'rb') as f:
    roi = pickle.load(f)

In [ ]:
with open(registration_base_path + '/color_scalebar_dict.pickle', 'rb') as file:
    scalebar_dict = pickle.load(file)

In [ ]:
roi_barcode_visualize = False

In [ ]:
if(roi_barcode_visualize):
    with open(pixel_barcode_base_path + '/roi_for_barcode_visualization.pickle', 'rb') as f:
        roi_barcode = pickle.load(f)

In [ ]:
fig, ax = plt.subplots(len(roi.keys()), len(stain_images))
fig.set_figheight(15)
fig.set_figwidth(15)

i = 0
for name, coord in roi.items():
    j = 0 
    for stain in stain_images:
        ax[i, j].imshow(stain[coord[0]:coord[1], coord[2]:coord[3]])
        ax[i, j].axis('off')
        scalebar = ScaleBar(scalebar_dict[stains[j]], 'um')
        ax[i, j].add_artist(scalebar)
        j+=1 
    i+=1 
    

plt.subplots_adjust(wspace=0.025, hspace=0.025) 

In [ ]:
if(roi_barcode_visualize):
    fig, ax = plt.subplots(len(roi_barcode.keys()), len(stain_images))
    fig.set_figheight(15)
    fig.set_figwidth(15)

    i = 0
    for name, sub_coord in roi_barcode.items():
        j = 0 
        coord = roi[name.split('_')[0]]
        for stain in stain_images:
            stain_roi_barcode = stain[coord[0]:coord[1], coord[2]:coord[3]].copy()
            stain_roi_barcode = cv2.rectangle(stain_roi_barcode, (list(roi_barcode.values())[i][0]-coord[0], list(roi_barcode.values())[i][2]-coord[2]), (list(roi_barcode.values())[i][1]-coord[0], list(roi_barcode.values())[i][3]-coord[2]), (0, 0, 0), 10)
            ax[i, j].imshow(np.rot90(stain_roi_barcode, k=1))
            ax[i, j].axis('off')
            scalebar = ScaleBar(scalebar_dict[stains[j]], 'um')
            ax[i, j].add_artist(scalebar)
            j+=1 
        i+=1 
        

    plt.subplots_adjust(wspace=0.025, hspace=0.025) 

In [ ]:
roi_visualized = np.zeros((1000, 1000, 4))
coord = roi['B']
for name, coord in roi.items():
    roi_visualized = cv2.rectangle(roi_visualized, (list(roi_barcode.values())[2][0]-coord[0], list(roi_barcode.values())[2][2]-coord[2]), (list(roi_barcode.values())[2][1]-coord[0], list(roi_barcode.values())[2][3]-coord[2]), (255, 255, 255, 255), 10)

In [ ]:
cv2.imwrite(registration_base_path + '/roi_B_barcode_visualized.png', roi_visualized)